# Batch Integration Metrics (scIB)

Run `scib-metrics` Benchmarker on each (encoder, view) configuration with
`batch=source` and `label=perturbation`. Reports both pre-Harmony (`X_pca`)
and post-Harmony (`X_pca_harmony`) so the contribution of batch correction
is visible.

This eval runs on the **full dataset**, not the held-out subset: scIB
measures whether batch correction successfully integrated the source
batches the model has, and that question is well-posed only when the data
contains multiple batches. The held-out generalization question is
addressed by `01_perturbation_recall/`.


In [ ]:
from __future__ import annotations

import os
os.environ["JAX_PLATFORMS"] = "cuda"  # GPU JAX → ~5x speedup over CPU on Benchmarker

from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scib_metrics.benchmark import Benchmarker, BatchCorrection, BioConservation


## Configuration

In [ ]:
EMBED_DIR = Path("../data")  # run notebooks from evals/<eval_name>/
DATASET = "Rxrx3C"

BATCH_KEY = "batch"
LABEL_KEY = "perturbation"
EMBEDDING_KEYS = ["X_pca", "X_pca_harmony"]
PRE_INTEGRATED_KEY = "X_pca"
N_JOBS = 16  # 20 CPUs affine on this SLURM node, leave a few for IO/GPU bookkeeping


## Configurations to evaluate

In [ ]:
ADATA_PATHS: dict[str, str] = {
    'DINO C': '25_Rxrx3C_DINO_ESM2_C_aggregated.h5ad',
    'DINO CD': '26_Rxrx3C_DINO_ESM2_CD_aggregated.h5ad',
    'DINO S': '27_Rxrx3C_DINO_ESM2_S_aggregated.h5ad',
    'DINO SD': '28_Rxrx3C_DINO_ESM2_SD_aggregated.h5ad',
    'OpenPhenom C': '29_Rxrx3C_OpenPhenom_ESM2_C_aggregated.h5ad',
    'OpenPhenom CD': '30_Rxrx3C_OpenPhenom_ESM2_CD_aggregated.h5ad',
    'OpenPhenom S': '31_Rxrx3C_OpenPhenom_ESM2_S_aggregated.h5ad',
    'OpenPhenom SD': '32_Rxrx3C_OpenPhenom_ESM2_SD_aggregated.h5ad',
    'SubCell C': '33_Rxrx3C_SubCell_ESM2_C_aggregated.h5ad',
    'SubCell CD': '34_Rxrx3C_SubCell_ESM2_CD_aggregated.h5ad',
    'SubCell S': '35_Rxrx3C_SubCell_ESM2_S_aggregated.h5ad',
    'SubCell SD': '36_Rxrx3C_SubCell_ESM2_SD_aggregated.h5ad',
}

## Run scIB benchmark

In [ ]:
bio = BioConservation(
    isolated_labels=True,
    nmi_ari_cluster_labels_kmeans=True,
    nmi_ari_cluster_labels_leiden=False,
    silhouette_label=True,
    clisi_knn=True,
)
batch = BatchCorrection(
    bras=True,
    ilisi_knn=True,
    kbet_per_label=True,
    graph_connectivity=True,
    pcr_comparison=True,
)

all_results: dict[str, pd.DataFrame] = {}
config_meta: dict[str, dict] = {}  # config -> {n_obs, n_perts, n_batches}

for label, fname in ADATA_PATHS.items():
    print(f"=== {label} ===")
    adata = ad.read_h5ad(EMBED_DIR / fname)
    adata.obs[BATCH_KEY] = adata.obs[BATCH_KEY].astype(str)
    adata.obs[LABEL_KEY] = adata.obs[LABEL_KEY].astype(str)
    config_meta[label] = {
        "n_obs": int(adata.n_obs),
        "n_perts": int(adata.obs[LABEL_KEY].nunique()),
        "n_batches": int(adata.obs[BATCH_KEY].nunique()),
    }
    print(
        f"  {config_meta[label]['n_obs']:,} wells, "
        f"{config_meta[label]['n_perts']} perturbations, "
        f"{config_meta[label]['n_batches']} batches ({BATCH_KEY})"
    )
    bm = Benchmarker(
        adata,
        batch_key=BATCH_KEY,
        label_key=LABEL_KEY,
        embedding_obsm_keys=EMBEDDING_KEYS,
        pre_integrated_embedding_obsm_key=PRE_INTEGRATED_KEY,
        bio_conservation_metrics=bio,
        batch_correction_metrics=batch,
        n_jobs=N_JOBS,
    )
    bm.benchmark()
    df = bm.get_results(min_max_scale=False, clean_names=True)
    df.index = pd.MultiIndex.from_tuples(
        [(label, emb) for emb in df.index], names=["config", "embedding"]
    )
    all_results[label] = df
    print(df.to_string())
    print()


## Combined results

In [ ]:
combined = pd.concat(all_results.values())
combined


## Harmony improvement (post − pre)

In [ ]:
pre = combined.xs("X_pca", level="embedding")
post = combined.xs("X_pca_harmony", level="embedding")
delta = post - pre
delta.columns = [f"Δ {c}" for c in delta.columns]
delta


## Heatmaps

In [ ]:
for embedding_key in EMBEDDING_KEYS:
    tag = "pre_harmony" if embedding_key == "X_pca" else "post_harmony"
    df = combined.xs(embedding_key, level="embedding").copy()
    df.index.name = "config"
    fig, ax = plt.subplots(figsize=(14, max(4, 0.4 * len(df))))
    sns.heatmap(
        df.astype(float),
        annot=True,
        fmt=".3f",
        cmap="YlOrRd",
        linewidths=0.5,
        ax=ax,
    )
    ax.set_title(f"scIB metrics — Rxrx3C ({tag})")
    ax.set_ylabel("")
    plt.tight_layout()
    plt.savefig(f"scib_heatmap_rxrx3c_{tag}.png", dpi=150, bbox_inches="tight")
    plt.show()


## Export CSVs (long-format)

In [ ]:
def _to_long(df: pd.DataFrame, has_embedding: bool) -> pd.DataFrame:
    out = df.reset_index()
    if has_embedding:
        # scib_metrics' get_results emits an extra "Metric Type" pseudo-row
        # alongside the real embeddings — drop it from CSV exports.
        out = out[out["embedding"].isin(EMBEDDING_KEYS)].copy()
    out.insert(0, "dataset", DATASET)
    out["n_obs"] = out["config"].map(lambda c: config_meta[c]["n_obs"])
    out["n_perts"] = out["config"].map(lambda c: config_meta[c]["n_perts"])
    out["n_batches"] = out["config"].map(lambda c: config_meta[c]["n_batches"])
    meta = ["dataset", "config"] + (["embedding"] if has_embedding else []) + ["n_obs", "n_perts", "n_batches"]
    metric_cols = [c for c in out.columns if c not in meta]
    return out[meta + metric_cols]


full_long = _to_long(combined, has_embedding=True)
delta_long = _to_long(delta, has_embedding=False)

full_csv = f"{DATASET}_scib_full.csv"
delta_csv = f"{DATASET}_scib_delta.csv"
full_long.to_csv(full_csv, index=False)
delta_long.to_csv(delta_csv, index=False)
print(f"wrote {full_csv}: {full_long.shape}")
print(f"wrote {delta_csv}: {delta_long.shape}")
full_long.head()


## Summary fragment for `SUMMARY.md`

In [ ]:
post = combined.xs("X_pca_harmony", level="embedding").astype(float)
delta_clean = delta.copy()
delta_clean.columns = [c.replace("Δ ", "") for c in delta_clean.columns]
delta_clean = delta_clean.astype(float)

n_obs_min = min(c["n_obs"] for c in config_meta.values())
n_obs_max = max(c["n_obs"] for c in config_meta.values())
n_perts = next(iter(config_meta.values()))["n_perts"]
n_batches = next(iter(config_meta.values()))["n_batches"]

lines: list[str] = [f"## {DATASET}", ""]
lines.append(
    f"**Configs:** {len(post)} | **Wells:** "
    f"{n_obs_min:,}–{n_obs_max:,} | **Batches:** {n_batches} | "
    f"**Perturbations:** {n_perts}"
)
lines.append("")
lines.append("**Best post-Harmony config per metric:**")
lines.append("")
for m in post.columns:
    cfg = post[m].idxmax()
    lines.append(f"- `{m}`: **{cfg}** (post-Harmony = {post.loc[cfg, m]:.3f})")
lines.append("")
lines.append("**Mean Harmony delta (post − pre):**")
lines.append("")
for m in delta_clean.columns:
    lines.append(
        f"- `{m}`: Δ={delta_clean[m].mean():+.3f} ± {delta_clean[m].std():.3f}"
    )
lines.append("")
lines.append(
    "**seg-vs-crops vs retrieval (placeholder — fill in once retrieval winner is computed):**"
)
lines.append("")

frag = "\n".join(lines)
frag_path = f"{DATASET}_scib_summary_fragment.md"
Path(frag_path).write_text(frag)
print(f"wrote {frag_path}")
print()
print(frag)
